### DATA Prepping

In [21]:
!pip install --quiet presidio-analyzer

In [11]:
import json

In [12]:
from utils import *

In [13]:
df_full = pd.read_csv('PII43k.csv', on_bad_lines='skip')

df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME_1] to send ...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [FULLNAME_1] who wants...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."
3,13. Write a press release announcing [FULLNAME...,13. Write a press release announcing Gayle Wat...,"['13', '.', 'write', 'a', 'press', 'release', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAM..."
4,9. Develop an inventory management plan for [F...,9. Develop an inventory management plan for Ev...,"['9', '.', 'develop', 'an', 'inventory', 'mana...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FU..."


In [14]:
cleaned_matches, unique_matches = get_template_tokens(df_full)

def replace_unique_tokens(text, tokens):
	for token in tokens:
		# Match either an underscore with one or more digits or with 'N'
		pattern = r'\[' + token + r'_(?:\d+|N)\]'
		# Replace with the token in square brackets (e.g., "[NAME]")
		text = re.sub(pattern, f'[{token}]', text)
	return text

df_full['Template'] = df_full['Template'].apply(lambda t: replace_unique_tokens(t, cleaned_matches))

# view the first 5 rows of the 'Template' column
df_full['Template'].head()

0    In our video conference, discuss the role of e...
1    Could you draft a letter for [NAME] to send to...
2    Discuss the options for [FULLNAME] who wants t...
3    13. Write a press release announcing [FULLNAME...
4    9. Develop an inventory management plan for [F...
Name: Template, dtype: object

In [15]:
df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME] to send to...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [FULLNAME] who wants t...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."
3,13. Write a press release announcing [FULLNAME...,13. Write a press release announcing Gayle Wat...,"['13', '.', 'write', 'a', 'press', 'release', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAM..."
4,9. Develop an inventory management plan for [F...,9. Develop an inventory management plan for Ev...,"['9', '.', 'develop', 'an', 'inventory', 'mana...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FU..."


In [16]:
cleaned_matches, unique_matches = get_template_tokens(df_full)
print(unique_matches)
print(len(unique_matches))

{'SEXTYPE', 'CURRENCY', 'LITECOINADDRESS', 'AMOUNT', 'CREDITCARDISSUER', 'CURRENCYCODE', 'EMAIL', 'STREET', 'NUMBER', 'USERAGENT', 'URL', 'ACCOUNTNAME', 'CITY', 'PASSWORD', 'CREDITCARDCVV', 'STATE', 'CURRENCYSYMBOL', 'CURRENCYNAME', 'JOBDESCRIPTOR', 'JOBAREA', 'ORDINALDIRECTION', 'USERNAME', 'PIN', 'MAC', 'NEARBYGPSCOORDINATE', 'COUNTY', 'DISPLAYNAME', 'FIRSTNAME', 'JOBTYPE', 'JOBTITLE', 'GENDER', 'IP', 'SEX', 'IBAN', 'MASKEDNUMBER', 'IPV4', 'NAME', 'LASTNAME', 'IPV6', 'ETHEREUMADDRESS', 'ZIPCODE', 'BITCOINADDRESS', 'BUILDINGNUMBER', 'CREDITCARDNUMBER', 'SECONDARYADDRESS', 'STREETADDRESS', 'ACCOUNTNUMBER', 'BIC', 'FULLNAME'}
49


In [17]:

# Load the JSON mapping from file (update the filename/path as needed)
with open("replace_list.json", "r") as f:
	token_mapping = json.load(f)

def replace_tokens_with_category(text, mapping):
	for category, tokens in mapping.items():
		for token in tokens:
			# Replace tokens with a suffix (e.g., [CITY_1] or [CITY_N])
			pattern = r'\[' + token + r'_(?:\d+|N)\]'
			text = re.sub(pattern, f'[{category.upper()}]', text)
			# Replace tokens without a suffix (e.g., [CITY])
			pattern_no_suffix = r'\[' + token + r'\]'
			text = re.sub(pattern_no_suffix, f'[{category.upper()}]', text)
	return text

# Update the 'Template' column in df_full
df_full['Template'] = df_full['Template'].apply(lambda t: replace_tokens_with_category(t, token_mapping))
print(df_full['Template'].head())

_, unique_matches = get_template_tokens(df_full)
print(unique_matches)
print(len(unique_matches))

0    In our video conference, discuss the role of e...
1    Could you draft a letter for [NAME] to send to...
2    Discuss the options for [NAME] who wants to ob...
3    13. Write a press release announcing [NAME]'s ...
4    9. Develop an inventory management plan for [N...
Name: Template, dtype: object
{'CURRENCY', 'LITECOINADDRESS', 'CREDITCARDISSUER', 'CURRENCYCODE', 'EMAIL', 'USERAGENT', 'URL', 'ACCOUNTNAME', 'JOB', 'PASSWORD', 'LOCATION', 'CURRENCYSYMBOL', 'CURRENCYNAME', 'ORDINALDIRECTION', 'USERNAME', 'DISPLAYNAME', 'GENDER', 'IP', 'MASKED_NUMBER', 'IBAN', 'MASKEDNUMBER', 'NAME', 'ETHEREUMADDRESS', 'BITCOINADDRESS', 'CREDITCARDNUMBER', 'BIC'}
26


In [18]:
# List of tokens to remove
remove_tokens = [
	"ORDINALDIRECTION", "ACCOUNTNAME", "CURRENCYSYMBOL", 
	"CURRENCYNAME", "CURRENCY", "CURRENCYCODE", 
	"CREDITCARDISSUER", "ETHEREUMADDRESS", "LITECOINADDRESS", "BITCOINADDRESS"
]

def remove_labels(text, tokens):
	for token in tokens:
		# Match token in square brackets with an optional numeric or 'N' suffix
		pattern = r'\[' + token + r'(?:_(?:\d+|N))?\]'
		text = re.sub(pattern, '', text)
	return text

# Remove the specified tokens from both the Template and Filled Template columns
df_full['Template'] = df_full['Template'].apply(lambda t: remove_labels(t, remove_tokens))
df_full['Filled Template'] = df_full['Filled Template'].apply(lambda t: remove_labels(t, remove_tokens))

# Display a few rows to verify
_, unique_matches = get_template_tokens(df_full)
print(unique_matches)
print(len(unique_matches))

{'GENDER', 'IP', 'URL', 'MASKED_NUMBER', 'JOB', 'MASKEDNUMBER', 'IBAN', 'USERNAME', 'NAME', 'PASSWORD', 'CREDITCARDNUMBER', 'DISPLAYNAME', 'EMAIL', 'BIC', 'LOCATION', 'USERAGENT'}
16


In [19]:

# Create 10 datasets with test percentages from 10% to 100%
test_pcts = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]

# Build list of datasets
df_list = [get_test_set(df_full, pct, random_state=42) for pct in test_pcts]


datasets = {}
for name, ds in zip(test_pcts, df_list):
    # Create a dataset instance with default (empty) init values.
    result = dataset(name=name, dataset=ds)
    datasets[name] = result
datasets[0.1].dataset.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
9,Hey! I need some advice on how to enforce my i...,Hey! I need some advice on how to enforce my i...,"['hey', '!', 'i', 'need', 'some', 'advice', 'o...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
55,Please write an email to [NAME] at [EMAIL] dis...,Please write an email to Hazel Turcotte at Ann...,"['please', 'write', 'an', 'email', 'to', 'haze...","['O', 'O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FUL..."
60,Could you help me draft an email to [NAME] at ...,Could you help me draft an email to Wolff - Kr...,"['could', 'you', 'help', 'me', 'draft', 'an', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-NA..."
77,"During the video conference, provide [NAME] wi...","During the video conference, provide Lola Sanf...","['during', 'the', 'video', 'conference', ',', ...","['O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAME', '..."
82,Create a list of resources on civil rights law...,Create a list of resources on civil rights law...,"['create', 'a', 'list', 'of', 'resources', 'on...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


### NLP Implementation 

In [22]:
from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.8 MB/s eta 0:00:00m eta 0:00:010:00:03
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [23]:
# test Call analyzer to get results
results = analyzer.analyze(text="My phone number is 212-555-5555",
                           entities=["PHONE_NUMBER"],
                           language='en')
print(results)

[type: PHONE_NUMBER, start: 19, end: 31, score: 0.75]


In [24]:
def clean_text(text):
    """Clean and normalize text."""
    return str(text).strip()

def extract_entities(template, filled):
    """
    Extract entities by aligning a template (with placeholders) to the filled text.
    
    Assumes that the filled text is identical to the template except that each placeholder 
    (e.g. "[NAME_1]") has been replaced by the actual value.
    
    Returns:
        A list of tuples (start_char, end_char, label) for entities in the filled text.
    """
    entities = []
    i = 0  # pointer for template
    j = 0  # pointer for filled text

    while i < len(template) and j < len(filled):
        if template[i] == '[':
            # Found a placeholder in the template.
            closing = template.find(']', i)
            if closing == -1:
                break  # malformed template (no matching ])
            # Extract the raw placeholder, e.g. "[NAME_1]"
            # Remove the brackets and any trailing digits to get the label.
            label_raw = template[i+1:closing]    # e.g. "NAME_1"
            label = re.sub(r'_\d+', '', label_raw)  # e.g. becomes "NAME"
            
            # Determine the literal text that follows the placeholder in the template.
            next_i = closing + 1
            next_bracket = template.find('[', next_i)
            literal = template[next_i:] if next_bracket == -1 else template[next_i:next_bracket]
            
            # In the filled text, the actual entity value replaces the placeholder.
            # We assume that the literal following the placeholder appears unchanged.
            if literal:
                literal_index = filled.find(literal, j)
            else:
                literal_index = len(filled)
            
            if literal_index == -1:
                # If we cannot find the literal, assume the entity is the rest of the filled text.
                entity_start = j
                entity_end = len(filled)
                j = len(filled)
            else:
                entity_start = j
                entity_end = literal_index
                j = literal_index  # advance pointer j to the beginning of the literal
            
            entities.append((entity_start, entity_end, label))
            # Advance pointer i past the entire placeholder.
            i = closing + 1
        else:
            # For non-placeholder characters, assume they match between template and filled.
            if template[i] == filled[j]:
                i += 1
                j += 1
            else:
                # If there is a mismatch (e.g. extra whitespace), increment j.
                j += 1
    return entities


In [25]:
import pandas as pd
import random
import spacy
from spacy.training.example import Example
from spacy.util import minibatch


# TODO: Run later: i want to collect F1 score, accuracy, false postetives, false negatives etc. everything

def NLP_training(df):
    # Build initial training data by aligning each template with its filled version.
    raw_train_data = []
    for _, row in df.iterrows():
        template = clean_text(row['Template'])
        filled = clean_text(row['Filled Template'])
        entities = extract_entities(template, filled)
        if entities:
            # Each training example is a tuple: (text, {"entities": [(start, end, label), ...]})
            raw_train_data.append((filled, {"entities": entities}))

    # ------------------------------
    # Step 0.5: Re-align Entity Offsets to Token Boundaries
    # ------------------------------
    tokenizer_nlp = spacy.blank("en")
    aligned_train_data = []
    for text, annotation in raw_train_data:
        doc = tokenizer_nlp(text)
        new_entities = []
        for start, end, label in annotation["entities"]:
            # Use "expand" mode to adjust the span to token boundaries.
            span = doc.char_span(start, end, alignment_mode="expand")
            if span is not None:
                new_entities.append((span.start_char, span.end_char, label))
            else:
                # If alignment fails, you might choose to log or skip the entity.
                print(f"WARNING: Could not align entity '{text[start:end]}' in text: {text}")
        if new_entities:
            aligned_train_data.append((text, {"entities": new_entities}))
    # Use the aligned data for training.
    TRAIN_DATA = aligned_train_data

    # ------------------------------
    # Step 1: Split Data
    # ------------------------------
    # Here we use an 80/20 train/validation split.
    train_size = int(0.8 * len(TRAIN_DATA))
    train_data = TRAIN_DATA[:train_size]
    valid_data = TRAIN_DATA[train_size:]

    # ------------------------------
    # Step 2: Create and Configure the Model
    # ------------------------------
    nlp = spacy.blank("en")

    # Add a Named Entity Recognizer (NER) pipeline component if not already present.
    if "ner" not in nlp.pipe_names:
        ner = nlp.add_pipe("ner", last=True)
    else:
        ner = nlp.get_pipe("ner")

    # Add each entity label from the training data to the NER component.
    for _, annotations in train_data:
        for start, end, label in annotations["entities"]:
            ner.add_label(label)

    # ------------------------------
    # Step 3: Train the Model Using Batches with Dropout
    # ------------------------------
    optimizer = nlp.begin_training()
    n_iter = 20  # Number of epochs
    batch_size = 16

    for itn in range(n_iter):
        random.shuffle(train_data)
        batches = minibatch(train_data, size=batch_size)
        losses = {}
        for batch in batches:
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                examples.append(Example.from_dict(doc, annotations))
            nlp.update(examples, sgd=optimizer, drop=0.3, losses=losses)
        print(f"Iteration {itn + 1}/{n_iter} - Losses: {losses}")

    # ------------------------------
    # Step 4: Define Improved Masking Function
    # ------------------------------
    def mask_pii(text, model):
        """
        Mask detected entities in the text with their label names.
        
        Entities are replaced starting from the end of the text (to avoid offset issues).
        """
        doc = model(text)
        spans = [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]
        # Sort spans in reverse order of start index.
        spans = sorted(spans, key=lambda x: x[0], reverse=True)
        masked_text = text
        for start, end, label in spans:
            masked_text = masked_text[:start] + label + masked_text[end:]
        return masked_text

    # ------------------------------
    # Step 5: Evaluate and Print Combined Results
    # ------------------------------
    correct_texts = 0
    correct_labels = 0
    total_texts = 0
    total_labels = 0
    failed_labels = []

    print("\n=== Evaluation on Validation Data ===\n")
    for text, annotations in valid_data:
        masked_text = mask_pii(text, nlp)
        
        print("Original Text:")
        print(text)
        print("Masked Text:")
        print(masked_text)
        print("-" * 40)
        
        text_correct = True
        # Evaluate masking on each individual entity.
        for start, end, label in annotations["entities"]:
            total_labels += 1
            if label in masked_text:
                correct_labels += 1
            else:
                text_correct = False
                failed_labels.append((label, text[start:end]))
        
        if text_correct:
            correct_texts += 1
        total_texts += 1

    if failed_labels:
        print("\nFAILED MASKINGS:")
        for label, value in failed_labels:
            print(f"Label: {label}, Expected Value: {value}")
    else:
        print("\nAll entities were successfully masked in every text!")

    text_accuracy = correct_texts / total_texts if total_texts > 0 else 0
    label_accuracy = correct_labels / total_labels if total_labels > 0 else 0

    print(f"\nText Accuracy (all entities in a text masked correctly): {text_accuracy:.2%}")
    print(f"Label Accuracy (individual entity masking): {label_accuracy:.2%}")
    return text_accuracy, label_accuracy, failed_labels, nlp


In [26]:
import pickle

for key, ds_obj in datasets.items():
    print(f"Processing dataset with test percentage: {ds_obj.name}")
    text_acc, label_acc, failed_labels, nlp_model = NLP_training(ds_obj.dataset)
    ds_obj.text_accuracy = text_acc
    ds_obj.label_accuracy = label_acc
    ds_obj.nlp = nlp_model
    print(ds_obj)

    with open(f"dataset_{ds_obj.name}.pkl", "wb") as file:
        pickle.dump(ds_obj, file)
    print(f"Saved ds_obj to dataset_{ds_obj.name}.pkl")

Processing dataset with test percentage: 0.1


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

# Extract sorted list of dataset keys (test percentages)
sorted_keys = sorted(datasets.keys())

# Get accuracy values (convert to percentage)
text_accs = [datasets[k].text_accuracy * 100 for k in sorted_keys]
label_accs = [datasets[k].label_accuracy * 100 for k in sorted_keys]

# Create the plot
plt.figure(figsize=(8, 5))
plt.plot(sorted_keys, text_accs, marker='o', label='Text Accuracy')
plt.plot(sorted_keys, label_accs, marker='o', label='Label Accuracy')
plt.xlabel('Test Percentage')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Comparison Across Datasets')
plt.legend()
plt.grid(True)
plt.show()